# Aave V3.1 — feature addition

Loads the **transformed** frames from `transformed_data/`:

- `DF_common_1` — asset-level reserve rates/indexes, keyed on `(time_bucket, asset)`
- `DF_common_final` — protocol-level 6h time series, keyed on `time_bucket`

In [ ]:
# Load transformed data from transformed_data/
import pandas as pd
from pathlib import Path
from IPython.display import display
import adv_validation as adv

DATA_DIR = Path("transformed_data")
PREVIEW_ROWS = 10

DF_common_1 = pd.read_csv(DATA_DIR / "DF_common_1.csv")          # asset-level (time_bucket, asset)
DF_common_final = pd.read_csv(DATA_DIR / "DF_common_final.csv")  # protocol-level 6h series


print(f" {DF_common_final.shape[0]} rows x {DF_common_final.shape[1]} cols")
display(DF_common_final.head(PREVIEW_ROWS))

 368 rows x 46 cols


,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth,borrow_tx_count,...,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,sampled_user_count,account_data_call_count
0,2025-11-01 00:00:00.000 UTC,317,305,191,197,5.817267e+08,150906.828593,7.058171e+08,183097.265312,197,...,20,16,16,6.334549e+03,0.000000e+00,5.891131e+03,0.950000,0.930000,1.0,3.0
1,2025-11-01 06:00:00.000 UTC,391,291,213,171,8.982494e+07,23209.831528,8.199260e+07,21186.012501,185,...,19,16,9,1.857490e+08,9.730363e+07,4.564883e+07,0.796000,0.769600,1.0,1.0
2,2025-11-01 12:00:00.000 UTC,425,307,205,153,7.963634e+07,20522.630436,9.197376e+07,23702.000478,286,...,25,17,23,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-11-01 18:00:00.000 UTC,304,181,171,107,1.781438e+08,45918.055714,1.730435e+08,44603.394329,186,...,30,24,31,4.923167e+03,4.527922e+03,5.062364e+01,0.950000,0.930000,1.0,1.0
4,2025-11-02 00:00:00.000 UTC,252,177,144,96,8.335144e+07,21467.127778,5.078573e+07,13079.799820,157,...,13,8,22,6.195266e+07,3.248046e+07,1.519899e+07,0.898667,0.876533,2.0,3.0
5,2025-11-02 06:00:00.000 UTC,319,273,172,174,5.264851e+07,13533.610105,9.971098e+07,25631.323976,174,...,23,18,25,5.368513e+03,1.343379e+02,4.826504e+03,0.926000,0.905000,2.0,5.0
6,2025-11-02 12:00:00.000 UTC,371,283,199,167,1.091697e+08,28245.532125,9.485326e+07,24541.436744,226,...,34,33,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2025-11-02 18:00:00.000 UTC,294,224,168,131,1.685130e+08,43637.518199,1.465321e+08,37945.402275,176,...,20,13,25,4.916688e+03,4.522577e+03,4.994348e+01,0.950000,0.930000,1.0,1.0
8,2025-11-03 00:00:00.000 UTC,518,471,235,241,3.256958e+08,85624.403438,4.972195e+08,130719.047959,266,...,71,74,50,1.395521e+05,1.065452e+05,5.899184e+03,0.890000,0.867500,3.0,6.0
9,2025-11-03 06:00:00.000 UTC,660,723,321,348,2.604916e+08,70122.120636,5.884341e+08,158401.641034,335,...,56,56,56,3.761479e+04,2.269418e+04,7.192926e+03,0.818000,0.793000,3.0,5.0


In [3]:
DF_common_final.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,str
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64
supply_amount_value_usd,float64
supply_amount_value_eth,float64
withdrawal_amount_value_usd,float64
withdrawal_amount_value_eth,float64
borrow_tx_count,int64


In [ ]:
# additional liquidity metrics

DF_common_final["net_liquidity_flow_usd"] = DF_common_final["supply_amount_value_usd"] - DF_common_final["withdrawal_amount_value_usd"]
DF_common_final["net_liquidity_flow_eth"] = DF_common_final["supply_amount_value_eth"] - DF_common_final["withdrawal_amount_value_eth"]

# Supply/Withdrawal Ratio
DF_common_final["supply_withdrawal_ratio"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["withdrawal_amount_value_usd"]
# DF_common_final["supply_withdrawal_ratio_eth"] = DF_common_final["supply_amount_value_eth"] / DF_common_final["withdrawal_amount_value_eth"]

# Liquidity Growth Rate
DF_common_final["liquidity_growth_rate"] = (
    DF_common_final["net_liquidity_flow_usd"] - DF_common_final["net_liquidity_flow_usd"].shift(1)
) / DF_common_final["net_liquidity_flow_usd"].shift(1).abs()

# Average Supply Size
DF_common_final["avg_supply_size_usd"] = DF_common_final["supply_amount_value_usd"] / DF_common_final["supply_tx_count"]

# Average Withdrawal Size
DF_common_final["avg_withdrawal_size_usd"] = DF_common_final["withdrawal_amount_value_usd"] / DF_common_final["withdrawal_tx_count"]

new_cols = [
    "net_liquidity_flow_usd",
    "net_liquidity_flow_eth",
    "supply_withdrawal_ratio",
    # "supply_withdrawal_ratio_eth",
    "liquidity_growth_rate",
    "avg_supply_size_usd",
    "avg_withdrawal_size_usd",
]

DF_common_final[["time_bucket"] + new_cols]

In [ ]:
temp = adv.statistical_validation(DF_common_final, columns = new_cols, save=False)
display(temp[["column", "null_pct", "zero_pct", "negative_pct",]])

In [ ]:
# additional borrow metrics
#avg_collateral_base col has null values, so adjusted for it
DF_common_final["net_borrow_demand_usd"] = DF_common_final["borrow_amount_value_usd"] - DF_common_final["repay_amount_value_usd"]
DF_common_final["net_borrow_demand_eth"] = DF_common_final["borrow_amount_value_eth"] - DF_common_final["repay_amount_value_eth"]

DF_common_final["borrow_repay_ratio"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["repay_amount_value_usd"].replace(0, pd.NA)

DF_common_final["avg_borrow_size_usd"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)
DF_common_final["avg_borrow_size_eth"] = DF_common_final["borrow_amount_value_eth"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

DF_common_final["avg_repay_size_usd"] = DF_common_final["repay_amount_value_usd"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)
DF_common_final["avg_repay_size_eth"] = DF_common_final["repay_amount_value_eth"] / DF_common_final["repay_tx_count"].replace(0, pd.NA)

DF_common_final["borrow_growth"] = (
    DF_common_final["borrow_amount_value_usd"] - DF_common_final["borrow_amount_value_usd"].shift(1)
) / DF_common_final["borrow_amount_value_usd"].shift(1).replace(0, pd.NA)

DF_common_final["debt_expansion_ratio"] = DF_common_final["avg_total_debt_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

borrow_cols = [
    "net_borrow_demand_usd",
    "net_borrow_demand_eth",
    "borrow_repay_ratio",
    "avg_borrow_size_usd",
    "avg_borrow_size_eth",
    "avg_repay_size_usd",
    "avg_repay_size_eth",
    "borrow_growth",
    "debt_expansion_ratio",
]

DF_common_final[["time_bucket"] + borrow_cols]

In [ ]:
temp_1 = adv.statistical_validation(DF_common_final, columns = borrow_cols, save=False)
display(temp_1[["column", "null_pct", "zero_pct", "negative_pct",]])

In [ ]:
#null value columns are used so expecting approximately same pct of null values
# implemented correction for NULL and zero values so there is no division by zero or such things

DF_common_final["collateralization_ratio"] = DF_common_final["avg_total_collateral_base"] / DF_common_final["avg_total_debt_base"].replace(0, pd.NA)

DF_common_final["borrow_capacity_utilization"] = DF_common_final["avg_total_debt_base"] / (
    DF_common_final["avg_total_debt_base"] + DF_common_final["avg_available_borrows_base"]
).replace(0, pd.NA)

DF_common_final["remaining_borrow_capacity"] = DF_common_final["avg_available_borrows_base"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)

DF_common_final["risk_buffer"] = DF_common_final["avg_current_liquidation_threshold"] - DF_common_final["avg_ltv"]

DF_common_final["ltv_utilization"] = DF_common_final["avg_ltv"] / DF_common_final["avg_current_liquidation_threshold"].replace(0, pd.NA)

DF_common_final["distance_to_liquidation"] = 1 - DF_common_final["ltv_utilization"]

risk_cols = [
    "collateralization_ratio",
    "borrow_capacity_utilization",
    "remaining_borrow_capacity",
    "risk_buffer",
    "ltv_utilization",
    "distance_to_liquidation",
]

DF_common_final[["time_bucket"] + risk_cols]

In [ ]:
temp_2 = adv.statistical_validation(DF_common_final, columns = risk_cols, save=False)
display(temp_2[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
#liquidation metrics
# the cols use the highest null val cols 
DF_common_final["liquidation_rate"] = DF_common_final["liquidation_tx_count"] / DF_common_final["borrow_tx_count"].replace(0, pd.NA)

# DF_common_final["liquidation_volume_ratio_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)
DF_common_final["liquidation_volume_ratio_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)

DF_common_final["liquidation_severity_usd"] = DF_common_final["liquidated_collateral_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
DF_common_final["liquidation_severity_eth"] = DF_common_final["liquidated_collateral_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["avg_liquidation_debt_usd"] = DF_common_final["liquidation_debt_covered_value_usd"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)
DF_common_final["avg_liquidation_debt_eth"] = DF_common_final["liquidation_debt_covered_value_eth"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["liquidator_concentration"] = DF_common_final["unique_liquidators"] / DF_common_final["liquidation_tx_count"].replace(0, pd.NA)

DF_common_final["liquidation_user_ratio"] = DF_common_final["unique_liquidated_users"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

liquidation_cols = [
    "liquidation_rate",
    # "liquidation_volume_ratio_usd" 
    "liquidation_volume_ratio_eth",
    "liquidation_severity_usd", "liquidation_severity_eth",
    "avg_liquidation_debt_usd", "avg_liquidation_debt_eth",
    "liquidator_concentration",
    "liquidation_user_ratio",
]

DF_common_final[["time_bucket"] + liquidation_cols]

In [ ]:
temp_3 = adv.statistical_validation(DF_common_final, columns = liquidation_cols, save=False)
display(temp_3[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
#user metrics, though sampled user are less. So not that effective as other values.

DF_common_final["borrower_participation_rate"] = DF_common_final["unique_borrowers"] / (
    DF_common_final["unique_borrowers"] + DF_common_final["unique_suppliers"]
).replace(0, pd.NA)

DF_common_final["repayment_discipline"] = DF_common_final["unique_repayers"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["supplier_activity"] = DF_common_final["supply_tx_count"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

DF_common_final["borrower_activity"] = DF_common_final["borrow_tx_count"] / DF_common_final["unique_borrowers"].replace(0, pd.NA)

DF_common_final["collateral_usage_rate"] = DF_common_final["collateral_enabled_count"] / (
    DF_common_final["collateral_enabled_count"] + DF_common_final["collateral_disabled_count"]
).replace(0, pd.NA)

DF_common_final["collateral_adoption_rate"] = DF_common_final["unique_collateral_enable_users"] / DF_common_final["unique_suppliers"].replace(0, pd.NA)

user_cols= [
    "borrower_participation_rate",
    "repayment_discipline",
    "supplier_activity",
    "borrower_activity",
    "collateral_usage_rate",
    "collateral_adoption_rate",
]

DF_common_final[["time_bucket"] + user_cols]

In [ ]:
temp_4 = adv.statistical_validation(DF_common_final, columns = user_cols, save=False)
display(temp_4[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
# values will be variably sparse , because flashloan tx count col hads 47.5% of null values
DF_common_final["avg_flashloan_size_usd"] = DF_common_final["flashloan_amount_value_usd"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)
DF_common_final["avg_flashloan_size_eth"] = DF_common_final["flashloan_amount_value_eth"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

DF_common_final["flashloan_fee_rate"] = DF_common_final["flashloan_premium_value_usd"] / DF_common_final["flashloan_amount_value_usd"].replace(0, pd.NA)

DF_common_final["flashloan_usage_intensity"] = DF_common_final["flashloan_tx_count"] / (
    DF_common_final["borrow_tx_count"] + DF_common_final["supply_tx_count"]
).replace(0, pd.NA)

DF_common_final["flashloan_user_activity"] = DF_common_final["flashloan_tx_count"] / DF_common_final["unique_flashloan_initiators"].replace(0, pd.NA)

DF_common_final["variable_debt_flashloan_ratio"] = DF_common_final["variable_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)  # ⚠️ 47.5% nulls

DF_common_final["no_debt_flashloan_ratio"] = DF_common_final["no_open_debt_flashloan_tx_count"] / DF_common_final["flashloan_tx_count"].replace(0, pd.NA)

flashloand_cols = [
    "avg_flashloan_size_usd", "avg_flashloan_size_eth",
    "flashloan_fee_rate",
    "flashloan_usage_intensity",
    "flashloan_user_activity",
    "variable_debt_flashloan_ratio",
    "no_debt_flashloan_ratio",
]

DF_common_final[["time_bucket"] + flashloand_cols]

In [ ]:
temp_5 = adv.statistical_validation(DF_common_final, columns = flashloand_cols, save=False)
display(temp_5[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
tx_cols = [
    "supply_tx_count", "withdrawal_tx_count", "borrow_tx_count",
    "repay_tx_count", "flashloan_tx_count", "liquidation_tx_count"
]

DF_common_final["total_activity"] = DF_common_final[tx_cols].fillna(0).sum(axis=1)

user_cols = ["unique_suppliers", "unique_borrowers", "unique_flashloan_initiators"]
DF_common_final["user_activity"] = DF_common_final[user_cols].fillna(0).sum(axis=1)

turnover_usd_cols = ["supply_amount_value_usd", "withdrawal_amount_value_usd", "borrow_amount_value_usd", "repay_amount_value_usd"]
turnover_eth_cols = ["supply_amount_value_eth", "withdrawal_amount_value_eth", "borrow_amount_value_eth", "repay_amount_value_eth"]
DF_common_final["protocol_turnover_usd"] = DF_common_final[turnover_usd_cols].fillna(0).sum(axis=1)
DF_common_final["protocol_turnover_eth"] = DF_common_final[turnover_eth_cols].fillna(0).sum(axis=1)

DF_common_final["leverage_indicator"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["supply_amount_value_usd"].replace(0, pd.NA)

DF_common_final["market_stress_index_usd"] = (
    DF_common_final["liquidation_tx_count"] *
    DF_common_final["liquidation_debt_covered_value_usd"]
) / DF_common_final["borrow_amount_value_usd"].replace(0, pd.NA)  # ⚠️ 47% nulls
DF_common_final["market_stress_index_eth"] = (
    DF_common_final["liquidation_tx_count"] *
    DF_common_final["liquidation_debt_covered_value_eth"]
) / DF_common_final["borrow_amount_value_eth"].replace(0, pd.NA)  # ⚠️ 47% nulls

DF_common_final["capital_efficiency"] = DF_common_final["borrow_amount_value_usd"] / DF_common_final["avg_total_collateral_base"].replace(0, pd.NA)  # ⚠️ 21% nulls

market_cols = [
    "total_activity",
    "user_activity",
    "protocol_turnover_usd", "protocol_turnover_eth",
    "leverage_indicator",
    "market_stress_index_usd", "market_stress_index_eth",
    "capital_efficiency",
]

DF_common_final[["time_bucket"] + market_cols]

In [ ]:
temp_6 = adv.statistical_validation(DF_common_final, columns = market_cols, save=False)
display(temp_6[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [4]:
print(f" {DF_common_1.shape[0]} rows x {DF_common_1.shape[1]} cols")
display(DF_common_1.head(PREVIEW_ROWS))

 6164 rows x 9 cols


,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,0.003290,0.000108,0.003290,1.000123,1.002229,2
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,NaN,0.038052,0.053091,1.012935,1.018159,6
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.002061,0.000038,0.001939,1.003386,1.024057,74
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.052500,0.000000,0.052500,1.000000,1.159328,7
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.047933,0.024595,0.047721,1.056670,1.108488,29
5,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,0.005480,0.000154,0.005479,1.000811,1.010175,19
6,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,0.055724,0.035723,0.055735,1.142631,1.215562,21
7,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,0.026826,0.009715,0.026826,1.063794,1.128767,33
8,2025-11-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,0.001951,0.000198,0.001952,1.001433,1.011235,24
9,2025-11-01 00:00:00.000 UTC,0x8292bb45bf1ee4d140127049757c2e0ff06317ed,RLUSD,0.042602,0.008186,0.042520,1.003996,1.019977,12


In [5]:
DF_common_1.dtypes.rename("dtype").to_frame()

,dtype
time_bucket,str
asset,str
asset_symbol,str
last_borrow_rate,float64
liquidity_rate,float64
variable_borrow_rate,float64
liquidity_index,float64
variable_borrow_index,float64
update_count,int64


In [17]:
#Rate dynamics, many cols have null values, adjusting for that.
DF_common_1["borrow_supply_spread"] = DF_common_1["last_borrow_rate"] - DF_common_1["liquidity_rate"]  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["variable_supply_spread"] = DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]

DF_common_1["borrow_rate_premium"] = DF_common_1["last_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls + 5.9% zeros guarded

DF_common_1["variable_rate_premium"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

# DF_common_1["rate_change"] = DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)  # ⚠️ 11.6% nulls
DF_common_1["rate_change"] = (DF_common_1["last_borrow_rate"] - DF_common_1["last_borrow_rate"].shift(1)) / DF_common_1["last_borrow_rate"].replace(0, pd.NA)

# DF_common_1["liquidity_rate_change"] = DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)
DF_common_1["liquidity_rate_change"] = (DF_common_1["liquidity_rate"] - DF_common_1["liquidity_rate"].shift(1)) / DF_common_1["liquidity_rate"].replace(0, pd.NA)

DF_common_1["variable_borrow_rate_change"] = DF_common_1["variable_borrow_rate"] - DF_common_1["variable_borrow_rate"].shift(1)

rate_dynamics_cols = [
    "borrow_supply_spread",
    "variable_supply_spread",
    "borrow_rate_premium",
    "variable_rate_premium",
    "rate_change",
    "liquidity_rate_change",
    "variable_borrow_rate_change",
]

DF_common_1[["time_bucket"] + rate_dynamics_cols]

,time_bucket,borrow_supply_spread,variable_supply_spread,borrow_rate_premium,variable_rate_premium,rate_change,liquidity_rate_change,variable_borrow_rate_change
0,2025-11-01 00:00:00.000 UTC,0.003181,0.003181,30.398993,30.398993,NaN,NaN,NaN
1,2025-11-01 00:00:00.000 UTC,NaN,0.015039,NaN,1.395226,NaN,0.997156,0.049802
2,2025-11-01 00:00:00.000 UTC,0.002024,0.001901,54.828494,51.573872,NaN,-1011.130313,-0.051152
3,2025-11-01 00:00:00.000 UTC,0.052500,0.052500,<NA>,<NA>,0.960737,<NA>,0.050561
4,2025-11-01 00:00:00.000 UTC,0.023338,0.023126,1.948875,1.940279,-0.095286,1.0,-0.004779
...,...,...,...,...,...,...,...,...
6159,2026-01-31 18:00:00.000 UTC,0.003844,0.003844,71.502883,71.502678,-0.343315,0.872024,-0.001338
6160,2026-01-31 18:00:00.000 UTC,0.013583,0.013583,1.519066,1.519073,0.901928,0.997916,0.035854
6161,2026-01-31 18:00:00.000 UTC,NaN,0.050668,NaN,10.428667,NaN,-3.869631,0.016290
6162,2026-01-31 18:00:00.000 UTC,0.018205,0.018199,3.794251,3.793361,NaN,0.175171,-0.031328


In [18]:
temp_7 = adv.statistical_validation(DF_common_1, columns = rate_dynamics_cols, save=False)
display(temp_7[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

,column,null_pct,zero_pct,mean,cv,p95
0,borrow_supply_spread,11.6483,0.0,1.788591e-02,1.789053,0.046935
1,variable_supply_spread,0.0000,0.0,1.584973e-02,0.844508,0.046200
2,borrow_rate_premium,17.4562,0.0,2.624234e+08,71.327206,165.682967
3,variable_rate_premium,5.9215,0.0,8.838657e+03,22.655457,170.298021
4,rate_change,21.9338,0.0,-1.473900e+03,23.181628,0.961806
5,liquidity_rate_change,5.9377,0.0,-6.806833e+09,62.993305,1.000000
6,variable_borrow_rate_change,0.0162,0.0,5.665918e-06,6209.553511,0.047887


In [ ]:
# growth metrics — per-asset shift(1) (DF_common_1 is asset-level, so t-1 is the same asset's prev 6h bucket)
# index cols are >= 1.0 (no zeros / nulls), so no denominator guard needed
DF_common_1["liquidity_index_growth"] = (
    DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
) / DF_common_1.groupby("asset")["liquidity_index"].shift(1)

DF_common_1["variable_borrow_index_growth"] = (
    DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)
) / DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

# interest / debt accrual velocity = per-asset delta of the index
DF_common_1["interest_accrual_velocity"] = DF_common_1["liquidity_index"] - DF_common_1.groupby("asset")["liquidity_index"].shift(1)
DF_common_1["debt_accrual_velocity"] = DF_common_1["variable_borrow_index"] - DF_common_1.groupby("asset")["variable_borrow_index"].shift(1)

growth_cols = [
    "liquidity_index_growth",
    "variable_borrow_index_growth",
    "interest_accrual_velocity",
    "debt_accrual_velocity",
]

DF_common_1[["time_bucket"] + growth_cols]

In [ ]:
temp_8 = adv.statistical_validation(DF_common_1, columns = growth_cols, save=False)
display(temp_8[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
# rate risk metrics — same-unit ratios (no usd/eth split)
# liquidity_rate 5.9% zeros guarded, last_borrow_rate 11.6% nulls
DF_common_1["borrow_pressure"] = DF_common_1["variable_borrow_rate"] / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["lending_attractiveness"] = DF_common_1["liquidity_rate"] / DF_common_1["last_borrow_rate"].replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["reserve_stress_score"] = (
    DF_common_1["variable_borrow_rate"] - DF_common_1["liquidity_rate"]
) / DF_common_1["liquidity_rate"].replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

rate_risk_cols = [
    "borrow_pressure",
    "lending_attractiveness",
    "reserve_stress_score",
]

DF_common_1[["time_bucket"] + rate_risk_cols]

In [ ]:
temp_9 = adv.statistical_validation(DF_common_1, columns = rate_risk_cols, save=False)
display(temp_9[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
# activity / volatility metrics — rolling std per asset over 4 × 6h (24h) windows
# update_count is already a metric, so not re-created here
DF_common_1["rate_volatility"] = DF_common_1.groupby("asset")["last_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_volatility"] = DF_common_1.groupby("asset")["liquidity_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

DF_common_1["borrow_volatility"] = DF_common_1.groupby("asset")["variable_borrow_rate"].transform(lambda s: s.rolling(4, min_periods=2).std())

volatility_cols = [
    "rate_volatility",
    "liquidity_volatility",
    "borrow_volatility",
]

DF_common_1[["time_bucket"] + volatility_cols]

In [ ]:
temp_10 = adv.statistical_validation(DF_common_1, columns = volatility_cols, save=False)
display(temp_10[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
# momentum (ML) features — per-asset rate(t) / rate(t-k); k in {1, 4} (4 × 6h = 24h), denominators guarded
DF_common_1["momentum"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(1).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["borrow_momentum_24h"] = DF_common_1["last_borrow_rate"] / DF_common_1.groupby("asset")["last_borrow_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["liquidity_momentum_24h"] = DF_common_1["liquidity_rate"] / DF_common_1.groupby("asset")["liquidity_rate"].shift(4).replace(0, pd.NA)  # ⚠️ 5.9% zeros guarded

DF_common_1["index_momentum_24h"] = DF_common_1["liquidity_index"] / DF_common_1.groupby("asset")["liquidity_index"].shift(4)  # index >= 1.0, no zero-guard needed

momentum_cols = [
    "momentum",
    "borrow_momentum_24h",
    "liquidity_momentum_24h",
    "index_momentum_24h",
]

DF_common_1[["time_bucket"] + momentum_cols]

In [ ]:
temp_11 = adv.statistical_validation(DF_common_1, columns = momentum_cols, save=False)
display(temp_11[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later

In [ ]:
# cross-asset features — rank within each time_bucket + asset share of updates (asset_symbol present)
DF_common_1["rank_by_borrow_rate"] = DF_common_1.groupby("time_bucket")["last_borrow_rate"].rank()  # ⚠️ 11.6% nulls from last_borrow_rate

DF_common_1["rank_by_liquidity_rate"] = DF_common_1.groupby("time_bucket")["liquidity_rate"].rank()

DF_common_1["asset_share_of_updates"] = DF_common_1["update_count"] / DF_common_1.groupby("time_bucket")["update_count"].transform("sum").replace(0, pd.NA)

cross_asset_cols = [
    "rank_by_borrow_rate",
    "rank_by_liquidity_rate",
    "asset_share_of_updates",
]

DF_common_1[["time_bucket"] + cross_asset_cols]

In [ ]:
temp_12 = adv.statistical_validation(DF_common_1, columns = cross_asset_cols, save=False)
display(temp_12[["column", "null_pct", "zero_pct", "mean", "cv", "p95",]])
# values and their analytics to be dealt later